In [ ]:
from libs import *
vosclient = Client()

In [ ]:
!vls vos:cfis/Processed_catalogues/StellarClass/stargal.cfis.r.dr5/tile.cats/

In [ ]:
def get_sg_fits(tile: str) -> str:
    """
    Acquires the tile.sg.fits file

    Parameters:
        tile (str): Name of the tile of interest.

    Returns:
        str: The path of the .fits file.
    """

    source = f"vos:cfis/Processed_catalogues/StellarClass/stargal.cfis.r.dr5/tile.cats/{tile}.sg.fits"

    local_cache = os.path.join(os.getcwd(), "star_gal")
    os.makedirs(local_cache, exist_ok=True)

    local_fit = os.path.join(local_cache, f"{tile}.sg.fits")
    
    if not os.path.exists(local_fit):
        vosclient.copy(source, local_fit)
        print(f"Downloaded {tile}.fits")
    else:
        print(f"{tile}.fits Already in folder")

    return local_fit

def extract_stars_per_tile(vos_source: str, output_dir: str, cache_dir: str):
    
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(exist_ok=True)
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    print("Listing remote files...")
    all_files = vosclient.listdir(vos_source)
    sg_files = [f for f in all_files if f.endswith(".sg.fits")]
    
    if not sg_files:
        print(f"No .sg.fits files found at {vos_source}")
        return
    
    print(f"Found {len(sg_files)} tiles to process...")
    
    total_stars = 0

    for i, filename in enumerate(sg_files):
        local_file = cache_dir / filename
        tile_name = filename.replace(".sg.fits", "")
        tile_csv = output_dir / f"{tile_name}.csv"

        # skip if already processed
        if tile_csv.exists():
            if (i + 1) % 10 == 0:
                print(f"  Skipped {i + 1}/{len(sg_files)} (already exists)...")
            continue

        try:
            vosclient.copy(vos_source + filename, str(local_file))
            
            with fits.open(local_file) as hdul:
                data = hdul[1].data
                df = pd.DataFrame(
                    np.array(data).byteswap().view(np.array(data).dtype.newbyteorder('='))
                ).astype(float)
            
            valid = df[(df["MAG_COG"] < 24)]

            stars = valid[
                ((valid["s21"] <= 3) & (valid["s31"] <= 3)) |
                (valid["MAG_COG"] < 17.5)
            ].copy()
            stars["TILE"] = tile_name
            stars.to_csv(tile_csv, index=False)

            total_stars += len(stars)

            if (i + 1) % 10 == 0:
                print(f"  Processed {i + 1}/{len(sg_files)} tiles, {len(stars)} stars in {tile_name}...")

        except Exception as e:
            print(f"  Warning: skipped {filename}: {e}")
        
        finally:
            if local_file.exists():
                local_file.unlink()
    
    print(f"\nDone! {total_stars:,} total stars across {len(sg_files)} tiles written to {output_dir}/")

source = "vos:cfis/Processed_catalogues/StellarClass/stargal.cfis.r.dr5/tile.cats/"
extract_stars_per_tile(source, output_dir="unions_stars", cache_dir="star_gal")

In [ ]:
def gaia_to_unions_tile(tile_name: str) -> str:
    return tile_name.replace("CFIS_LSB.", "CFIS.")

def match_gaia_to_tiles(gaia_csv: str, 
                        tiles_dir: str = "unions_stars",
                        matches_out: str = "gaia_unions_matches.csv",
                        sep_arcsec: float = 10.0):

    tiles_dir = Path(tiles_dir)
    matches_path = Path(matches_out)
    completed_log = Path("unions_stars/completed_tiles.txt")

    print("Loading GAIA catalogue...")
    gaia = pd.read_csv(gaia_csv)#, index_col=0)
    gaia = gaia.drop(columns=[col for col in gaia.columns if 'Unnamed' in col])
    gaia = gaia[(gaia['star_ra'] != -999.0) & (gaia['star_dec'] != -999.0)]
    gaia = gaia.dropna(subset=['star_ra', 'star_dec'])

    print(f"GAIA stars (clean): {len(gaia):,}")

    gaia_by_tile = gaia.groupby('tile_name')
    print(f"Found {len(gaia_by_tile)} GAIA tiles to process...")

    total_matched = 0
    total_gaia_added = 0

    for i, (tile_name, gaia_tile) in enumerate(gaia_by_tile):

        try:
            unions_tile_name = tile_name.replace("CFIS_LSB.", "CFIS.")
            tile_file = tiles_dir / f"{unions_tile_name}.csv"
    
            if completed_log.exists():
                completed = set(completed_log.read_text().splitlines())
                if tile_file.name in completed:
                    print(f"{tile_file.name} already processed.")
                    continue
    
            if not tile_file.exists():
                print(f"  Warning: {tile_file.name} not found (from {tile_name}), skipping.")
                continue

            unions = pd.read_csv(tile_file)
            if unions.empty:
                continue

            gaia_tile = gaia_tile.copy().reset_index(drop=True)

            # coordinates
            unions_coords = SkyCoord(
                ra=unions['ALPHA_J2000'].values * u.degree,
                dec=unions['DELTA_J2000'].values * u.degree
            )
            gaia_coords = SkyCoord(
                ra=gaia_tile['star_ra'].values * u.degree,
                dec=gaia_tile['star_dec'].values * u.degree
            )

            idx, sep, _ = gaia_coords.match_to_catalog_sky(unions_coords)
            match_mask = sep.arcsec <= sep_arcsec

            # save matches
            if match_mask.sum() > 0:
                gaia_matched   = gaia_tile[match_mask].copy().reset_index(drop=True)
                unions_matched = unions.iloc[idx[match_mask]].copy().reset_index(drop=True)

                gaia_matched   = gaia_matched.add_prefix('gaia_')
                unions_matched = unions_matched.add_prefix('unions_')

                matches = pd.concat([gaia_matched, unions_matched], axis=1)
                matches['sep_arcsec'] = sep.arcsec[match_mask]

                matches.to_csv(matches_path, mode='a',
                               header=not matches_path.exists(), index=False)

            # update tile
            unions['source'] = 'unions'
            unions['matched'] = False

            if match_mask.sum() > 0:
                unions.loc[idx[match_mask], 'matched'] = True

            gaia_tile_out = gaia_tile.rename(columns={
                'star_ra':    'ALPHA_J2000',
                'star_dec':   'DELTA_J2000',
                'star_g_mag': 'gaia_g_mag'
            })
            gaia_tile_out['source'] = 'gaia'
            gaia_tile_out['matched'] = match_mask

            combined = pd.concat([unions, gaia_tile_out], ignore_index=True)
            combined.to_csv(tile_file, index=False)

            total_matched += match_mask.sum()
            total_gaia_added += len(gaia_tile)

            if (i + 1) % 100 == 0:
                print(f"  Processed {i + 1}/{len(gaia_by_tile)} GAIA tiles — {total_matched:,} matches")

            with open(completed_log, 'a') as f:
                f.write(tile_file.name + '\n')

        except Exception as e:
            print(f"  Warning: skipped {tile_name}: {e}")

    print("\nDone!")
    print(f"Total matches         : {total_matched:,}")
    print(f"Total GAIA processed  : {total_gaia_added:,}")

match_gaia_to_tiles(
    "gaia2_not_in_gaia1.csv",
    "unions_stars",
    "gaia_unions_matches.csv",
    10.0
)